# Настройка DuckLake

In [1]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [2]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml or /Users/i.korsakov/.jupysql/config.

Did not find user configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml.

In [3]:
%sql duckdb:///:memory:

Connecting and switching to connection 'duckdb:///:memory:'

# Создание подключения к DuckLake

In [4]:
%%sql
INSTALL ducklake;
INSTALL postgres;

,Success


In [5]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE postgres,
    HOST 'localhost',
    PORT 5432,
    DATABASE postgres,
    USER 'postgres',
    PASSWORD 'postgres'
);

,Success


In [6]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE s3,
    URL_STYLE 'path',
    USE_SSL FALSE,
    ENDPOINT 'localhost:9000',
    KEY_ID 'minioadmin',
    SECRET 'minioadmin'
);

,Success


In [7]:
%%sql
ATTACH 'ducklake:postgres:dbname=postgres' AS my_ducklake (DATA_PATH 's3://prod/ducklake/');

,Success


In [8]:
%%sql
USE my_ducklake;

,Success


# Создание таблицы в DuckLake

In [9]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

,Success


In [10]:
%%sql
FROM fake_data

,id,name,email,city,country
0,1,Bobbie Rolfson,reannatreutel@auer.info,Jastmouth,Czech Republic
1,2,Sim Klocko,catharinewalker@kemmer.name,Jadynland,Tajikistan
2,3,Helena Krajcik,eloyking@wiza.org,Wittingside,Macao
3,4,Rylan Wuckert,parkermcclure@schaefer.org,Simoneburgh,Turks and Caicos Islands
4,5,Delpha Ondricka,aurelioskiles@sporer.net,Kingfort,Lebanon
...,...,...,...,...,...
95,96,Allen Rodriguez,ivaleffler@kling.name,Leoneview,Senegal
96,97,Ryann Miller,alfschuppe@kozey.biz,Goodwinborough,Mauritania
97,98,Clare Aufderhar,charleygoyette@roberts.info,Vilmaton,Mongolia
98,99,Beaulah Cole,myrtlepfannerstill@nader.name,Christiansenside,Yemen


# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [11]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

,Success


In [13]:
%%sql
FROM fake_data

,id,name,email,city,country,name_prefix
0,1,Bobbie Rolfson,reannatreutel@auer.info,Jastmouth,Czech Republic,None
1,2,Sim Klocko,catharinewalker@kemmer.name,Jadynland,Tajikistan,None
2,3,Helena Krajcik,eloyking@wiza.org,Wittingside,Macao,None
3,4,Rylan Wuckert,parkermcclure@schaefer.org,Simoneburgh,Turks and Caicos Islands,None
4,5,Delpha Ondricka,aurelioskiles@sporer.net,Kingfort,Lebanon,None
...,...,...,...,...,...,...
95,96,Allen Rodriguez,ivaleffler@kling.name,Leoneview,Senegal,None
96,97,Ryann Miller,alfschuppe@kozey.biz,Goodwinborough,Mauritania,None
97,98,Clare Aufderhar,charleygoyette@roberts.info,Vilmaton,Mongolia,None
98,99,Beaulah Cole,myrtlepfannerstill@nader.name,Christiansenside,Yemen,None


In [14]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

,Success


In [15]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Bobbie Rolfson,reannatreutel@auer.info,Jastmouth,Czech Republic,Mr.
1,2,Sim Klocko,catharinewalker@kemmer.name,Jadynland,Tajikistan,Mr.
2,3,Helena Krajcik,eloyking@wiza.org,Wittingside,Macao,Dr.
3,4,Rylan Wuckert,parkermcclure@schaefer.org,Simoneburgh,Turks and Caicos Islands,Ms.
4,5,Delpha Ondricka,aurelioskiles@sporer.net,Kingfort,Lebanon,Ms.
...,...,...,...,...,...,...
95,96,Allen Rodriguez,ivaleffler@kling.name,Leoneview,Senegal,Mr.
96,97,Ryann Miller,alfschuppe@kozey.biz,Goodwinborough,Mauritania,Dr.
97,98,Clare Aufderhar,charleygoyette@roberts.info,Vilmaton,Mongolia,Miss
98,99,Beaulah Cole,myrtlepfannerstill@nader.name,Christiansenside,Yemen,Dr.


# Time travel

In [16]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [17]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [19]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,0,"created_schema:""main""",None,None,None
1,1,"created_table:""main"".""fake_data"",inserted_into...",None,None,None
2,2,altered_table:1,None,None,None
3,3,"inserted_into_table:1,deleted_from_table:1",None,None,None


In [20]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-06-02 10:49:52.022826+03:00,0,1,0
1,1,2026-06-02 10:49:58.624035+03:00,1,2,1
2,2,2026-06-02 10:50:00.411892+03:00,2,2,1
3,3,2026-06-02 10:50:30.504315+03:00,2,2,2


In [21]:
%%sql
USE 'my_ducklake';

,Success


In [24]:
%%sql
SELECT * FROM fake_data AT (VERSION => 3);

,id,name,email,city,country,name_prefix
0,1,Bobbie Rolfson,reannatreutel@auer.info,Jastmouth,Czech Republic,Mr.
1,2,Sim Klocko,catharinewalker@kemmer.name,Jadynland,Tajikistan,Mr.
2,3,Helena Krajcik,eloyking@wiza.org,Wittingside,Macao,Dr.
3,4,Rylan Wuckert,parkermcclure@schaefer.org,Simoneburgh,Turks and Caicos Islands,Ms.
4,5,Delpha Ondricka,aurelioskiles@sporer.net,Kingfort,Lebanon,Ms.
...,...,...,...,...,...,...
95,96,Allen Rodriguez,ivaleffler@kling.name,Leoneview,Senegal,Mr.
96,97,Ryann Miller,alfschuppe@kozey.biz,Goodwinborough,Mauritania,Dr.
97,98,Clare Aufderhar,charleygoyette@roberts.info,Vilmaton,Mongolia,Miss
98,99,Beaulah Cole,myrtlepfannerstill@nader.name,Christiansenside,Yemen,Dr.
